In [4]:
#always run this first!!
#essential reticulate functions that allow us to use python packages in R
#you'll need a conda env with 'reticulate', leidenalg' and 'pandas' installed to do this
#then route reticulate to the python installed in that conda env with the below functions
#change the path to whichever conda env you installed reticulate in
#Sys.setenv(RETICULATE_PYTHON="/.conda/envs/reticulate/bin/python")
#library(reticulate)
#reticulate::use_python("/.conda/envs/reticulate/bin/python")
#reticulate::use_condaenv("/.conda/envs/reticulate")
#reticulate::py_module_available(module='leidenalg') #needs to be TRUE
#reticulate::import('leidenalg') #good to make sure this doesn't error

In [5]:
#source activate newEnv
suppressMessages(library(hdf5r))
suppressMessages(library(Seurat))
suppressMessages(library(Signac))
suppressMessages(library(EnsDb.Hsapiens.v86))
suppressMessages(library(dplyr))
suppressMessages(library(ggplot2))
suppressMessages(library(Matrix))
suppressMessages(library(harmony))
suppressMessages(library(data.table))
suppressMessages(library(ggpubr))
suppressMessages(library(future))
library(dplyr)
library(ggplot2)
library(sctransform)
#library(scater)
library(reticulate)
library(future)
#library('gPCA')
library('Biobase')
#library(pheatmap)
#library("ggfortify")
#library('qvalue')
library(gplots)
#library('DESeq2')
#library(VennDiagram)
library('hdf5r')
library(EnsDb.Hsapiens.v86)
#library(EnsDb.Mmusculus.v79)
library(BiocParallel)
#library(tictoc)
library(Seurat)
library(Signac)
library(EnsDb.Hsapiens.v86)
library(ggplot2)
library(cowplot)
library(GenomeInfoDb)

library("Signac")


Warning message:
“package ‘hdf5r’ was built under R version 4.2.2”

Attaching package: 'gplots'


The following object is masked from 'package:IRanges':

    space


The following object is masked from 'package:S4Vectors':

    space


The following object is masked from 'package:stats':

    lowess



Attaching package: 'cowplot'


The following object is masked from 'package:ggpubr':

    get_legend




In [6]:
##DO THIS OR IT WILL RANDOMLY PUT YOUR COORDINATES IN SCIENTIFIC NOTATION
options(scipen=999)

In [7]:
#Read in peak lists
cells<-scan('/marker_CREs/012924_markerCREs_unionPeaks/celltypes.txt', sep="\n", what="")
orgpeaks<-list()
for (cell in cells){
    orgpeaks[[cell]]<-read.table(sprintf('/marker_CREs/012924_markerCREs_unionPeaks/%s.noheader.bed',cell), sep="\t", header=TRUE)
}
head(orgpeaks[[1]])

,chr,start,end,length,abs_summit,pileup,X.log10.pvalue.,fold_enrichment,X.log10.qvalue.,name
,<chr>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<chr>
1,chr1,9920,10679,760,10111,250,250.1370,23.19780,247.4150,Acinar_1_2_6_peak_1
2,chr1,99529,99875,347,99693,104,79.9046,12.48900,77.5638,Acinar_1_2_6_peak_2
3,chr1,180670,181600,931,180850,107,72.9764,10.36470,70.6608,Acinar_1_2_6_peak_3
4,chr1,184160,184632,473,184445,75,38.4347,6.66667,36.2755,Acinar_1_2_6_peak_4
5,chr1,267842,268167,326,268002,223,239.1450,26.64320,236.4410,Acinar_1_2_6_peak_5
6,chr1,276132,276454,323,276307,134,116.2180,16.05730,113.7670,Acinar_1_2_6_peak_6


In [8]:
#Add in headers if you read in a headerless version
orgpeaks2<-lapply(cells, function(c){
    df<-orgpeaks[[c]]
    colnames(df)<-c('Chr', 'Start','End', 'Size','abs_summit', "pileup", "-log10(pvalue)","fold_enrichment","-log10(qvalue)","name")
    return(df)
})

In [9]:
head(orgpeaks2[[1]])

,Chr,Start,End,Size,abs_summit,pileup,-log10(pvalue),fold_enrichment,-log10(qvalue),name
,<chr>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<chr>
1,chr1,9920,10679,760,10111,250,250.1370,23.19780,247.4150,Acinar_1_2_6_peak_1
2,chr1,99529,99875,347,99693,104,79.9046,12.48900,77.5638,Acinar_1_2_6_peak_2
3,chr1,180670,181600,931,180850,107,72.9764,10.36470,70.6608,Acinar_1_2_6_peak_3
4,chr1,184160,184632,473,184445,75,38.4347,6.66667,36.2755,Acinar_1_2_6_peak_4
5,chr1,267842,268167,326,268002,223,239.1450,26.64320,236.4410,Acinar_1_2_6_peak_5
6,chr1,276132,276454,323,276307,134,116.2180,16.05730,113.7670,Acinar_1_2_6_peak_6


In [11]:
#Shrinks peaks over 300bp to 300bp centered around summit
shrunkpeaks<-lapply(1:length(cells), function(i){
    df<-orgpeaks2[[i]]
    for (i in 1:nrow(df)){
        psize<-df$Size[i]
        pstart<-df$Start[i]
        pend<-df$End[i]
        if (psize > 300){
            summit<-df$abs_summit[i]
            newstart<-as.numeric(summit)-150
            newend<-as.numeric(summit)+150
            df$Start[i]<-newstart
            df$End[i]<-newend
            df$Size[i]<-newend-newstart
        }

    }
    return(df)
})

In [15]:
#Just making sure things are in the right order before saving with celltype name
##THIS PART DEPENDS ON YOUR PEAK NAMING CONVENTION!!! Might be different from mine which is Celltype_peak_#
for (i in 1:length(cells)){
     df<-shrunkpeaks[[i]]
    string<-df[1,10]
    delimiter <- "_peak"

    # Split the string based on the delimiter
    split_string <- strsplit(string, delimiter)[[1]]
    
    # Extract everything before the delimiter
    result <- split_string[1]

    ctorder<-cells[i]
    message(paste0("Does ", result, " match ", ctorder,"?"))
}

Does Acinar_1_2_6 match Acinar_1_2_6?

Does Acinar_3 match Acinar_3?

Does Acinar_5 match Acinar_5?

Does Acinar_4 match Acinar_4?

Does Ductal match Ductal?

Does Activated_Stellate match Activated_Stellate?

Does Endothelial match Endothelial?

Does Schwann match Schwann?

Does Alpha match Alpha?

Does Delta match Delta?

Does Beta match Beta?

Does Macrophage match Macrophage?

Does Tcells match Tcells?

Does Mast match Mast?

Does Quiescent_Stellate match Quiescent_Stellate?

Does LymphEndo match LymphEndo?



In [16]:
#Adds in the cell type as names so you can refer back to each df by name
names(shrunkpeaks)<-cells

In [17]:
#Saving it out
for (i in 1:length(cells)){
    df<-shrunkpeaks[[i]]
    ct<-cells[i]
    write.table(df, sprintf('/marker_CREs/012924_markerCREs_unionPeaks/CorrectedUnionPeaks_052724/%s.shrunkpeaks.bed',ct), row.names=FALSE, sep="\t", quote=FALSE)
}